In [31]:
import pandas as pd

In [32]:
df1 = pd.read_csv('data/movie/tmdb_5000_credits.csv')
df2 = pd.read_csv('data/movie/tmdb_5000_movies.csv')

In [33]:
df1.rename(columns={'movie_id':'id'}, inplace=True)
df1.columns

Index(['id', 'title', 'cast', 'crew'], dtype='object')

In [34]:
df_temp = df1[['id', 'cast', 'crew']]
df_temp.columns

Index(['id', 'cast', 'crew'], dtype='object')

In [35]:
df = df2.merge(df_temp, on='id')
df.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'cast', 'crew'],
      dtype='object')

In [36]:
df['overview'].head(5)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: overview, dtype: object

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

In [38]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
ENGLISH_STOP_WORDS

frozenset({'a',
           'about',
           'above',
           'across',
           'after',
           'afterwards',
           'again',
           'against',
           'all',
           'almost',
           'alone',
           'along',
           'already',
           'also',
           'although',
           'always',
           'am',
           'among',
           'amongst',
           'amoungst',
           'amount',
           'an',
           'and',
           'another',
           'any',
           'anyhow',
           'anyone',
           'anything',
           'anyway',
           'anywhere',
           'are',
           'around',
           'as',
           'at',
           'back',
           'be',
           'became',
           'because',
           'become',
           'becomes',
           'becoming',
           'been',
           'before',
           'beforehand',
           'behind',
           'being',
           'below',
           'beside',
           'besides'

In [39]:
df['overview'].isnull().sum()

np.int64(3)

In [40]:
df['overview'].fillna('', inplace=True)

C:\Users\AIPM2\AppData\Local\Temp\ipykernel_7260\2799063249.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['overview'].fillna('', inplace=True)


In [41]:
df['overview'].isnull().sum()

np.int64(0)

In [42]:
len(df['overview'])

4803

In [43]:
#BOW 생성
tfidf_matrix = tfidf.fit_transform(df['overview'])

In [44]:
tfidf_matrix.shape

(4803, 20978)

In [45]:
#문장 유사도
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [46]:
cosine_sim.shape

(4803, 4803)

In [47]:
cosine_sim

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.02160533, 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.01488159, 0.        ,
        0.        ],
       ...,
       [0.        , 0.02160533, 0.01488159, ..., 1.        , 0.01609091,
        0.00701914],
       [0.        , 0.        , 0.        , ..., 0.01609091, 1.        ,
        0.01171696],
       [0.        , 0.        , 0.        , ..., 0.00701914, 0.01171696,
        1.        ]], shape=(4803, 4803))

In [48]:
df['title'].head()

0                                      Avatar
1    Pirates of the Caribbean: At World's End
2                                     Spectre
3                       The Dark Knight Rises
4                                 John Carter
Name: title, dtype: object

In [49]:
title='John Carter'
idx=df[df['title']==title].index[0]
idx

np.int64(4)

In [50]:
cosine_sim[4]

array([0.        , 0.03336868, 0.        , ..., 0.00612609, 0.        ,
       0.        ], shape=(4803,))

In [51]:
test_consine_sim = list(enumerate(cosine_sim[4]))
test_consine_sim = sorted(test_consine_sim, key=lambda x:x[1], reverse=True)
test_consine_sim=test_consine_sim[1:11]
index = [i[0] for i in test_consine_sim]
index

[1254, 4161, 2932, 3349, 1307, 3068, 345, 581, 2998, 4274]

In [52]:
df.loc[index, 'title']

1254                          Get Carter
4161         The Marine 4: Moving Target
2932                        Raising Cain
3349                           Desperado
1307                       The Hurricane
3068                         Rescue Dawn
345                          Rush Hour 2
581              Star Trek: Insurrection
2998                               Devil
4274    Eddie: The Sleepwalking Cannibal
Name: title, dtype: object

In [53]:
def def_cosine_sim():
    #BOW생성
    from sklearn.feature_extraction.text import TfidfVectorizer
    tfidf = TfidfVectorizer(stop_words='english')
    
    df = pd.read_csv('data/movie/tmdb_5000_movies.csv')
    df.fillna({'overview':''}, inplace=True)
    tfidf_matrix = tfidf.fit_transform(df['overview'])

    #문장 유사도
    from sklearn.metrics.pairwise import linear_kernel
    cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
    return cosine_sim

In [54]:
cosine_sim = def_cosine_sim()
cosine_sim

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.02160533, 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.01488159, 0.        ,
        0.        ],
       ...,
       [0.        , 0.02160533, 0.01488159, ..., 1.        , 0.01609091,
        0.00701914],
       [0.        , 0.        , 0.        , ..., 0.01609091, 1.        ,
        0.01171696],
       [0.        , 0.        , 0.        , ..., 0.00701914, 0.01171696,
        1.        ]], shape=(4803, 4803))

In [55]:
df[['title']].head()

,title
0,Avatar
1,Pirates of the Caribbean: At World's End
2,Spectre
3,The Dark Knight Rises
4,John Carter


In [56]:
title='The Dark Knight Rises'
idx=df[df['title']==title].index[0]
print(idx)

3


In [57]:
sim = cosine_sim[idx]
title_sim = list(enumerate(sim))
title_sim = sorted(title_sim, key=lambda x:x[1], reverse=True)
title_sim = title_sim[1:11]
index = [x[0] for x in title_sim]
df.loc[index, 'title']

65                              The Dark Knight
299                              Batman Forever
428                              Batman Returns
1359                                     Batman
3854    Batman: The Dark Knight Returns, Part 2
119                               Batman Begins
2507                                  Slow Burn
9            Batman v Superman: Dawn of Justice
1181                                        JFK
210                              Batman & Robin
Name: title, dtype: object

In [58]:
def recommend(title):
    import pickle

    df = pd.read_csv('data/movie/tmdb_5000_movies.csv')
    idx=df[df['title']==title].index[0]

    cosine_sim = pickle.load(open('data/movie/cosine_sim.pickle', 'rb'))
    sim = cosine_sim[idx]

    sim = list(enumerate(sim))
    sim = sorted(sim, key=lambda x: x[1], reverse=True)
    sim = sim[1:11]
    index = [x[0] for x in sim]
    return index

In [59]:
index = recommend('Avatar')
index

[3604, 2130, 634, 1341, 529, 1610, 311, 847, 775, 2628]

In [60]:
idx = recommend('Batman Forever')
df.loc[idx, 'title']

3                         The Dark Knight Rises
119                               Batman Begins
65                              The Dark Knight
428                              Batman Returns
210                              Batman & Robin
3854    Batman: The Dark Knight Returns, Part 2
1359                                     Batman
4343                                   Cry_Wolf
174                         The Incredible Hulk
9            Batman v Superman: Dawn of Justice
Name: title, dtype: object

In [61]:
cosine_sim = def_cosine_sim()
cosine_sim[:3]

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.02160533, 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.01488159, 0.        ,
        0.        ]], shape=(3, 4803))

In [62]:
import pickle
pickle.dump(cosine_sim, open('data/movie/cosine_sim.pickle', 'wb'))

In [63]:
#tmdb api (영화)
df=pd.read_csv('data/movie/movies.csv')
df.loc['id','title']

FileNotFoundError: [Errno 2] No such file or directory: 'data/movie/movies.csv'

In [ ]:
from tmdbv3api import Movie,TMDb
tmdb=TMDb()
tmdb.api_key='c668cda4cf75bf267ef2aeffa2da0341'
tmdb.language='ko-KR'
movie=Movie()

details=movie.details('19995')
# print(details)
details=['title']
poster="https://image.tmdb.org/t/p/w500" + details['poster_path']
overview=details['overview']
print(title)
print(poster)
print(overview)


TypeError: list indices must be integers or slices, not str